In [2]:
import pandas as pd
import numpy as np
import h5py
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.dummy import DummyRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.model_selection import GridSearchCV

# # PyTorch RNN
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.preprocessing import StandardScaler
import numpy as np


Load and Explore the .h5 File

In [3]:
# open the file in read mode
# File name
fn = "data/data_for_ml.h5"

# Explore keys in the HDF5 file
with h5py.File(fn, 'r') as f:
    keys = list(f.keys()) #gives list of the names used in the file
    print("Available keys in the file:")
    for key in keys:
        print(key)


Available keys in the file:
DCIR
Dynamic_PsRP_1_1C
Dynamic_PsRP_1_C
Dynamic_PsRP_2_1C
Dynamic_PsRP_2_C
Static_HPPC
Static_PsRP_1
Static_PsRP_2_Chg
Static_PsRP_2_Dis
Static_Rapid


Load the Dataset

In [4]:
# loading specific data file 
#  fn = file name and key= specific column we want to open

data = pd.read_hdf(fn, key='Dynamic_PsRP_1_1C') 
print(data.shape)
data.head()


(4351, 264)


,soc,direction,voltage_0.0s,voltage_1.0s,voltage_2.0s,voltage_3.0s,voltage_4.0s,voltage_5.0s,voltage_6.0s,voltage_7.0s,...,Charge depleting cycle charge throughput,Charge sustaining cycle charge efficiency,Post C/10 charge relaxation fit MSE,Post C/5 charge relaxation fit MSE,Post C/3 charge relaxation fit MSE,Post P/3 charge relaxation fit MSE,Post C/2 charge relaxation fit MSE,Post 1C charge relaxation fit MSE,Thickness growth,Volume growth
0,0.011132,charge,6.799,7.092,7.119,7.142,7.163,6.624,6.608,6.599,...,53.193351,1.254012,0.000015,0.000014,0.000013,0.000013,0.000013,0.000013,NaN,NaN
1,0.172361,charge,7.966,8.200,8.209,8.216,8.222,7.760,7.749,7.742,...,53.193351,1.254012,0.000015,0.000014,0.000013,0.000013,0.000013,0.000013,NaN,NaN
2,0.333591,charge,8.126,8.341,8.350,8.357,8.363,7.936,7.925,7.917,...,53.193351,1.254012,0.000015,0.000014,0.000013,0.000013,0.000013,0.000013,NaN,NaN
3,0.494596,charge,8.288,8.411,8.411,8.410,8.411,8.098,8.089,8.082,...,53.193351,1.254012,0.000015,0.000014,0.000013,0.000013,0.000013,0.000013,NaN,NaN
4,0.644375,charge,8.387,8.408,8.409,8.410,8.410,8.192,8.183,8.177,...,53.193351,1.254012,0.000015,0.000014,0.000013,0.000013,0.000013,0.000013,NaN,NaN


Data Preparation

In [5]:
# Convert direction to numeric
data['direction_num'] = data['direction'].map({'charge': 1, 'discharge': 0})

# Extract cell_type from cell_id
data['cell_type'] = data['cell_id'].astype(str).str[0]

# Drop the columns
drop_cols = [
    'cell_id',
    'Unnamed: 0',
    'measurement_id',
    'direction',
    'Charge depleting cycle charge throughput',
    'Charge sustaining cycle charge efficiency',
    'Post C/10 charge relaxation fit MSE',
    'Post C/5 charge relaxation fit MSE',
    'Post C/3 charge relaxation fit MSE',
    'Post P/3 charge relaxation fit MSE',
    'Post C/2 charge relaxation fit MSE',
    'Post 1C charge relaxation fit MSE',
    'Thickness growth',
    'Volume growth']

new_data = data.drop(columns=drop_cols)

# Define target and exogenous variables
target_vars = [
    'C/10 discharge capacity', 'C/5 discharge capacity',
    'C/3 discharge capacity', 'C/2 discharge capacity',
    '1C discharge capacity', 'P/3 discharge capacity']

exogenous_vars = ['soc', 'temperature_ambient']


Filter for Cell Type 'C' and Prepare Features

In [6]:
# Filter for cell type 'C'
new_data_C = new_data[new_data['cell_type'] == 'A']

# Select numeric columns
numeric_C = new_data_C.select_dtypes(include='number')

# Define features and target
feature_cols = [col for col in numeric_C.columns if col not in target_vars]
X = numeric_C[feature_cols]
y = numeric_C['C/10 discharge capacity']

print("Feature shape:", X.shape)


Feature shape: (938, 245)


In [7]:
X.head()

,soc,voltage_0.0s,voltage_1.0s,voltage_2.0s,voltage_3.0s,voltage_4.0s,voltage_5.0s,voltage_6.0s,voltage_7.0s,voltage_8.0s,...,current_113.0s,current_114.0s,current_115.0s,current_116.0s,current_117.0s,current_118.0s,current_119.0s,current_120.0s,temperature_ambient,direction_num
349,0.008191,3.265,3.316,3.325,3.332,3.339,3.251,3.245,3.241,3.238,...,128.0,128.0,128.0,128.0,128.0,128.0,128.0,128.0,31.0,1
350,0.106692,3.570,3.607,3.609,3.611,3.612,3.539,3.537,3.535,3.533,...,128.0,128.0,128.0,128.0,128.0,128.0,128.0,128.0,31.0,1
351,0.205194,3.643,3.678,3.680,3.682,3.683,3.615,3.613,3.611,3.609,...,128.0,128.0,128.0,128.0,128.0,128.0,128.0,128.0,31.0,1
352,0.303695,3.691,3.724,3.725,3.727,3.728,3.662,3.660,3.658,3.657,...,128.0,128.0,128.0,128.0,128.0,128.0,128.0,128.0,31.0,1
353,0.402197,3.729,3.761,3.763,3.765,3.767,3.701,3.698,3.696,3.694,...,128.0,128.0,128.0,128.0,128.0,128.0,128.0,128.0,31.0,1


Define Model Evaluation Function